# Single-Query Molecular Retrieval

## Setup

In [ ]:
from setup import *

from matplotlib.legend_handler import HandlerBase
from matplotlib.lines import Line2D

## Compute Query Neighbors

In [ ]:
def query_dataset_stats(query_embedding, dataset_embeddings, query_fingerprint, dataset_fingerprints, num_buckets=20):
    buckets = np.linspace(-1, 1, num_buckets + 1) # fence post problem
    embedding_similarities = torch.nn.CosineSimilarity(dim=1)(query_embedding, dataset_embeddings).numpy()
    fingerprint_similarities = packed_jaccard_similarity(query_fingerprint, dataset_fingerprints)

    # naive solution:
    embedding_similarities_bucket_idx = np.digitize(embedding_similarities, buckets[1:-1])
    stats = []
    for i in range(num_buckets):
        fingerprint_bucket_vals = fingerprint_similarities[embedding_similarities_bucket_idx == i]
        if len(fingerprint_bucket_vals) == 0:
            stats.append({"domain": (buckets[i], buckets[i+1]),
                          "count": 0,
                          "q1": None, "median": None, "q3": None, "iqr": None,
                          "mean": None, "stdev": None})
            continue
        q1, median, q3 = torch.quantile(fingerprint_bucket_vals, torch.tensor([0.25, 0.5, 0.75]))
        stats.append({
            "domain": (buckets[i], buckets[i+1]),
            "count": len(fingerprint_bucket_vals),
            "q1": q1.item(), "median": median.item(), "q3": q3.item(), "iqr": (q3 - q1).item(),
            "mean": fingerprint_bucket_vals.mean().item(),
            "stdev": fingerprint_bucket_vals.std(unbiased=True).item() if fingerprint_bucket_vals.numel() > 1 else 0.0,
        })
    return buckets, stats

def compute_query_curve(model, query_smiles, embedding_cache, fingerprints):
    batch = smiles_to_batch([query_smiles], n_confs=1)
    query_embedding = model.encode(batch.to(model.device)).cpu()
    embeddings = embedding_cache["embeddings"].cpu()
    if not torch.equal(embedding_cache["id"], fingerprints["id"]): embeddings = embeddings[match_molecule_ids(embedding_cache["id"], fingerprints["id"])]
    curve = []
    for fp_type in ("ecfp", "rdkit"):
        query_fingerprint = torch.tensor(smiles_to_fingerprint(query_smiles, fp_type, packed=True))
        _, stats = query_dataset_stats(query_embedding, embeddings, query_fingerprint, fingerprints[fp_type])
        curve.extend([
            [values["domain"][0] for values in stats if values["count"] > 0],
            [values["median"] for values in stats if values["count"] > 0],
            [values["iqr"] for values in stats if values["count"] > 0],
        ])
    return tuple(curve)

query_smiles = "C1CC(N(C1)C(=O)C2=CC=CS2)C3=CC=C(C=C3)Cl"
retrieval_model_name = "ANT"
embedding_cache = load_or_compute_embeddings(EMBEDDING_DIR / f"embeddings_{retrieval_model_name}.pt", CHECKPOINTS[retrieval_model_name], CHEMBL_DIR, DEVICE)
fingerprints = load_or_compute_fingerprints(FINGERPRINT_DIR, CHEMBL_DIR, SMILES_DIR, ids=embedding_cache["id"])
id_to_smiles = dict(zip(fingerprints["id"].tolist(), fingerprints["smiles"]))
model = EncoderModel.load_model(CHECKPOINTS[retrieval_model_name], device=DEVICE).eval()
query_curve_ANT = compute_query_curve(model, query_smiles, embedding_cache, fingerprints)

nt_embedding_cache = load_or_compute_embeddings(EMBEDDING_DIR / "embeddings_NT.pt", CHECKPOINTS["NT"], CHEMBL_DIR, DEVICE)
nt_model = EncoderModel.load_model(CHECKPOINTS["NT"], device=DEVICE).eval()
query_curve_NT = compute_query_curve(nt_model, query_smiles, nt_embedding_cache, fingerprints)

topk = 20
nearest_neighbors_dict = retrieve_neighbors(model, [query_smiles], embedding_cache["embeddings"], molecule_ids=embedding_cache["id"], id_to_smiles=id_to_smiles, topk=topk)

## Figure S7: Fingerprint Similarity Across Retrieval Scores

In [ ]:
class HandlerErrorbarDot(HandlerBase):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        color = orig_handle.get_color()
        x = xdescent + width / 2
        y = ydescent + height / 2
        line = Line2D([x, x], [ydescent - height * 0.4, ydescent + height * 1.4], color=color, alpha=0.5, linewidth=0.6, transform=trans)
        cap1 = Line2D([x - 2, x + 2], [ydescent - height * 0.4, ydescent - height * 0.4], color=color, alpha=0.5, linewidth=0.6, transform=trans)
        cap2 = Line2D([x - 2, x + 2], [ydescent + height * 1.4, ydescent + height * 1.4], color=color, alpha=0.5, linewidth=0.6, transform=trans)
        dot = Line2D([x], [y], marker="o", markersize=3, markerfacecolor=color, markeredgecolor=color, linestyle="None", alpha=1, transform=trans)
        return [line, cap1, cap2, dot]

def plot_query_curve(ax, data, title):
    x_vals_ecfp, medians_ecfp, iqrs_ecfp, x_vals_rdkit, medians_rdkit, iqrs_rdkit = data
    ax.errorbar(x_vals_ecfp, medians_ecfp, yerr=iqrs_ecfp, fmt="o", color="#9e0142", capsize=2, elinewidth=0.6, alpha=0.5, markersize=3, markeredgecolor="#9e0142", markerfacecolor="#9e0142")
    ax.plot(x_vals_ecfp, medians_ecfp, "o", color="#9e0142", markersize=3)
    ax.errorbar(x_vals_rdkit, medians_rdkit, yerr=iqrs_rdkit, fmt="o", color="#3288bd", capsize=2, elinewidth=0.6, alpha=0.5, markersize=3, markeredgecolor="#3288bd", markerfacecolor="#3288bd")
    ax.plot(x_vals_rdkit, medians_rdkit, "o", color="#3288bd", markersize=3)
    ax.set_title(title, fontsize=6, pad=0)
    ax.set_xlim(-0.4, 1)
    ax.set_yticks(np.arange(0, 0.76, 0.25))
    ax.tick_params(labelsize=4, length=2, width=0.4, pad=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(0.4)

fig, axes = plt.subplots(2, 1, figsize=(3.5 * cm, 7.1 * cm), sharex=True)
plot_query_curve(axes[0], query_curve_ANT, MODEL_TITLES["ANT"])
axes[0].tick_params(axis="x", which="both", bottom=True, labelbottom=False)
axes[0].set_ylim(0.0, 0.75)
axes[0].set_xlim(-0.4, 1)

legend_handles = [
    Line2D([0], [0], marker="o", color="#9e0142", markerfacecolor="#9e0142", markeredgecolor="#9e0142", markersize=3, label="ECFP"),
    Line2D([0], [0], marker="o", color="#3288bd", markerfacecolor="#3288bd", markeredgecolor="#3288bd", markersize=3, label="RDKFP")
]

axes[0].legend(handles=legend_handles, handler_map={Line2D: HandlerErrorbarDot()}, frameon=False, fontsize=5, markerscale=0.5, loc="upper left", handlelength=1)
plot_query_curve(axes[1], query_curve_NT, f"{MODEL_TITLES['NT']} (Control)")
axes[1].set_ylim(0.0, 0.75)
axes[1].set_xlim(-0.4, 1)
fig.subplots_adjust(left=0.18, hspace=0.15, bottom=0.08, top=0.95, right=0.95)
plt.xticks(np.arange(-0.4, 1.1, 0.2))
fig.text(0.04, 0.5, "Fingerprint Similarity (Jaccard)", fontsize=6, rotation=90, ha="center", va="center")
axes[1].set_xlabel("Embedding Similarity (Cosine)", fontsize=6, ha="center", va="center")
fig.savefig(OUTPUT_DIR / "FP_embedding_scatter_combined.png", dpi=600)
plt.show()

## Retrieved Molecules

In [ ]:
display(pd.DataFrame([
    {"Query": query, "Rank": rank + 1, "SMILES": neighbor,
     "Cosine similarity": values["similarity"][rank],
     "Fingerprint similarity": values["fp_similarity"][rank]}
    for query, values in nearest_neighbors_dict.items()
    for rank, neighbor in enumerate(values["neighbor_smiles"])
]))